# 다음 24시간 강수 예측 · PyTorch
강남 AWS 지점 400에서 다음 24시간 중 측정 강수가 한 번이라도 있을 확률을 학습합니다.
학습 2015~2022년, 검증 2023~2024년입니다. 2025년은 최종 평가를 위해 남겨둡니다.
실습 노트북처럼 리소스 설정 → 데이터 준비 → 모델 설계 → 학습 → 검증 → 저장 순서로 실행하세요.
커널은 프로젝트의 `.venv`를 선택하세요. 필요한 라이브러리는 torch, pandas, numpy, scikit-learn, joblib입니다.
강수량은 비와 눈을 구분하지 않습니다. 표시하는 확률은 강수 발생 확률이며, 이번 예측이 맞을 확률과 같지 않습니다.

## 1. 라이브러리와 학습 리소스 설정

In [ ]:
import sys
import json
import runpy
from pathlib import Path
from datetime import datetime
from time import perf_counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, average_precision_score, brier_score_loss
)
import sklearn
import joblib

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
    print("GPU:", torch.cuda.get_device_name(0))
print("사용 디바이스:", device)
print("파이썬:", sys.executable)
print("PyTorch:", torch.__version__)


## 2. 전처리된 학습·검증 자료 연결
`weather_data_v2`를 프로젝트 폴더에 풀어놓았다면 자동으로 찾습니다. 현재 컴퓨터에서는 준비된 결과 폴더도 확인합니다.
이 단계에서는 2025년 테스트 파일을 읽지 않습니다. 입력 목록은 학습 후보에서만 정합니다.

In [ ]:
STATION = 400
candidates = [
    Path.cwd() / "weather_data_v2",
    Path.cwd(),
    Path(r"C:\Users\jeonm\Documents\Codex\2026-10-12\wlrm\outputs\weather_data_v2"),
]
DATA_DIR = None
for candidate in candidates:
    config_file = candidate / "model_input_columns.json"
    if config_file.exists():
        candidate_config = json.loads(config_file.read_text(encoding="utf-8"))
        if candidate_config.get("version") == "aws-split-v2":
            DATA_DIR, config = candidate, candidate_config
            break
if DATA_DIR is None:
    raise FileNotFoundError("새 데이터의 weather_data_v2 폴더를 프로젝트 폴더에 넣어주세요.")

helper = runpy.run_path(str(DATA_DIR / "check_data.py"))
feature_candidates = config["model_input_columns"]
assert not set(feature_candidates) & set(config["forbidden_input_columns"])
target = config["rain"]["target"]
eligible = config["rain"]["eligible"]
datasets = {}
for split in ("train", "validation"):
    spec = config["splits"][split]
    observations = helper["read_station"](
        DATA_DIR / spec["input_file"], STATION,
        list(dict.fromkeys(["station_id", "station_name", "observed_at", *feature_candidates]))
    )
    targets = helper["read_station"](
        DATA_DIR / spec["target_file"], STATION,
        ["station_id", "observed_at", "target_end_at", target, eligible]
    )
    joined = observations.merge(targets, on=["station_id", "observed_at"],
                                how="left", validate="one_to_one", indicator=True)
    assert joined["_merge"].eq("both").all()
    selected = joined.loc[joined[eligible].eq(1)].reset_index(drop=True)
    assert not selected.empty and selected[target].isin([0, 1]).all()
    assert selected.target_end_at.le(pd.Timestamp(spec["end_at"])).all()
    datasets[split] = selected

train, validation = datasets["train"], datasets["validation"]
feature_columns = [c for c in feature_candidates if train[c].notna().any()]
dropped_columns = [c for c in feature_candidates if c not in feature_columns]
X_train, y_train = train[feature_columns].copy(), train[target].astype("int8")
X_val, y_val = validation[feature_columns].copy(), validation[target].astype("int8")

print("데이터 폴더:", DATA_DIR)
print(f"학습 {len(y_train):,}행 / 검증 {len(y_val):,}행 / 입력 {len(feature_columns)}개")
print(f"학습 강수 비율 {y_train.mean():.2%} / 검증 강수 비율 {y_val.mean():.2%}")
print("학습에서 전부 결측인 입력 제외:", dropped_columns)


## 3. 결측 처리·정규화·데이터로더
결측을 채울 중앙값과 정규화 기준은 학습 자료에서만 구합니다.
검증에는 같은 처리를 적용합니다. 입력 결측 여부도 추가 입력으로 보존합니다.
Windows 주피터에서 데이터로더를 실행하도록 `num_workers=0`을 사용합니다.

In [ ]:
preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
])
X_train_ready = preprocessor.fit_transform(X_train).astype(np.float32)
X_val_ready = preprocessor.transform(X_val).astype(np.float32)
assert np.isfinite(X_train_ready).all() and np.isfinite(X_val_ready).all()

train_dataset = TensorDataset(
    torch.from_numpy(X_train_ready),
    torch.tensor(y_train.to_numpy(), dtype=torch.float32).reshape(-1, 1)
)
val_dataset = TensorDataset(
    torch.from_numpy(X_val_ready),
    torch.tensor(y_val.to_numpy(), dtype=torch.float32).reshape(-1, 1)
)
BATCH_SIZE = 256
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=0, pin_memory=(device.type == "cuda"),
    generator=torch.Generator().manual_seed(SEED)
)
val_loader = DataLoader(
    val_dataset, batch_size=1024, shuffle=False,
    num_workers=0, pin_memory=(device.type == "cuda")
)
print("신경망 입력 항목:", X_train_ready.shape[1])
print("학습 배치 수:", len(train_loader))


## 4. 강수 예측 모델 설계
기상 수치와 과거 특징을 입력으로 받는 신경망입니다. 출력 하나를 sigmoid로 변환하면 강수 발생 확률이 됩니다.
학습 손실은 sigmoid와 이진 손실을 함께 계산하는 BCEWithLogitsLoss입니다.
[PyTorch 손실 함수](https://docs.pytorch.org/docs/2.14/generated/torch.nn.BCEWithLogitsLoss.html)

In [ ]:
INPUT_SIZE = X_train_ready.shape[1]
HIDDEN_SIZES = [128, 64]
DROPOUT = 0.1

def make_model(input_size):
    return nn.Sequential(
        nn.Linear(input_size, HIDDEN_SIZES[0]),
        nn.ReLU(),
        nn.Dropout(DROPOUT),
        nn.Linear(HIDDEN_SIZES[0], HIDDEN_SIZES[1]),
        nn.ReLU(),
        nn.Linear(HIDDEN_SIZES[1], 1),
    )

model = make_model(INPUT_SIZE).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.0001)

print(model)
print("모델이 올라간 장치:", next(model.parameters()).device)


## 5. 학습 실행
모델과 각 배치를 같은 장치로 옮깁니다. 최대 20회 학습하며, 검증 손실이 5회 연속 개선되지 않으면 멈춥니다.
검증 손실이 가장 낮았던 가중치를 보관합니다. 정답 비율을 바꾸는 가중치·재표본추출은 적용하지 않습니다.
[PyTorch 학습 흐름](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html)

In [ ]:
def train_epoch():
    model.train()
    loss_sum = 0.0
    for inputs, labels in train_loader:
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(inputs)
        loss = criterion(logits, labels)
        if not torch.isfinite(loss):
            raise RuntimeError("학습 손실이 유효한 숫자가 아닙니다.")
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * len(labels)
    return loss_sum / len(train_dataset)

def validate_epoch():
    model.eval()
    loss_sum, probabilities = 0.0, []
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            logits = model(inputs)
            loss_sum += criterion(logits, labels).item() * len(labels)
            probabilities.append(torch.sigmoid(logits).cpu().numpy().ravel())
    return loss_sum / len(val_dataset), np.concatenate(probabilities)

EPOCHS = 20
PATIENCE = 5
best_val_loss, best_epoch, best_state = float("inf"), 0, None
stale_epochs, history = 0, []
started = perf_counter()

for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch()
    val_loss, probabilities = validate_epoch()
    val_accuracy = accuracy_score(y_val, probabilities >= 0.5)
    history.append({"epoch":epoch, "train_loss":train_loss,
                    "val_loss":val_loss, "val_accuracy":val_accuracy})
    if val_loss < best_val_loss:
        best_val_loss, best_epoch = val_loss, epoch
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        stale_epochs = 0
    else:
        stale_epochs += 1
    print(f"Epoch {epoch:02d} | Train loss {train_loss:.4f} | "
          f"Val loss {val_loss:.4f} | Val 정확도 {val_accuracy:.2%}", flush=True)
    if stale_epochs >= PATIENCE:
        print("검증 손실 개선이 멈춰 조기 종료합니다.")
        break

model.load_state_dict(best_state)
model.eval()
print(f"학습 완료: 선택 Epoch {best_epoch}, 소요 {perf_counter()-started:.1f}초")


## 6. 검증 성능과 확률 확인
표시는 우선 50% 기준입니다. 이 기준과 확률 보정은 검증으로 추가 확인합니다.
항상 강수 없음만 예측하는 경우와 학습 강수 비율만 출력하는 경우를 함께 비교합니다.
아래 예측 예시는 과거 검증 시점이며 현재 날씨 예보가 아닙니다. 2025년 최종 성능을 뜻하지 않습니다.

In [ ]:
_, val_prob = validate_epoch()
THRESHOLD = 0.5
val_pred = (val_prob >= THRESHOLD).astype("int8")
baseline_prob = np.full(len(y_val), y_train.mean())

metrics = {
    "accuracy":float(accuracy_score(y_val, val_pred)),
    "precision":float(precision_score(y_val, val_pred, zero_division=0)),
    "recall":float(recall_score(y_val, val_pred, zero_division=0)),
    "f1":float(f1_score(y_val, val_pred, zero_division=0)),
    "average_precision":float(average_precision_score(y_val, val_prob)),
    "brier":float(brier_score_loss(y_val, val_prob)),
    "always_dry_accuracy":float(y_val.eq(0).mean()),
    "constant_train_rate_brier":float(brier_score_loss(y_val, baseline_prob)),
    "threshold":THRESHOLD,
}
print(f"검증 정확도: {metrics['accuracy']:.2%}")
print(f"항상 강수 없음 정확도: {metrics['always_dry_accuracy']:.2%}")
print(classification_report(y_val, val_pred, labels=[0,1],
      target_names=["강수 없음", "강수 있음"], digits=3, zero_division=0))
print(f"평균 정밀도(AP): {metrics['average_precision']:.4f}")
print(f"확률 오차(Brier): {metrics['brier']:.4f}")
print(f"학습 강수 비율만 출력한 Brier: {metrics['constant_train_rate_brier']:.4f}")

examples = pd.DataFrame({
    "검증 기준 시각":validation.observed_at,
    "정답 구간 끝":validation.target_end_at,
    "강수 예상":np.where(val_pred==1, "강수 있음", "강수 없음"),
    "다음 24시간 강수 확률(%)":np.round(val_prob*100, 1),
    "실제 정답":y_val.to_numpy(),
})
print(examples.tail(5).to_string(index=False))


## 7. 모델·전처리·입력 규격 저장
프로젝트의 models 폴더 아래 새 실행 폴더에 저장합니다. GPU에서 학습한 가중치는 CPU 텐서로 저장해 API에서도 불러올 수 있도록 합니다.
API에서는 같은 입력 칼럼 순서와 전처리를 적용해야 합니다. 이 모델은 강남 지점만 지원하는 첫 후보이며 확률 보정은 아직 적용하지 않았습니다.

In [ ]:
RUN_DIR = Path.cwd() / "models" / f"rain_pytorch_{STATION}_{datetime.now():%Y%m%d_%H%M%S_%f}"
RUN_DIR.mkdir(parents=True, exist_ok=False)
torch.save(best_state, RUN_DIR / "model_weights.pt")
joblib.dump(preprocessor, RUN_DIR / "preprocessor.joblib")
history_df = pd.DataFrame(history)
history_df.to_csv(RUN_DIR / "training_history.csv", index=False, encoding="utf-8-sig")

model_config = {
    "station_id":STATION, "model_type":"MLP", "input_size":INPUT_SIZE,
    "hidden_sizes":HIDDEN_SIZES, "dropout":DROPOUT,
    "feature_columns":feature_columns, "dropped_columns":dropped_columns,
    "target":target, "target_hours":24, "threshold":THRESHOLD,
    "probability_calibrated":False, "best_epoch":best_epoch,
    "seed":SEED, "dataset_version":config["version"],
    "train_period":config["splits"]["train"],
    "validation_period":config["splits"]["validation"],
    "test_evaluated":False, "training_device":str(device),
    "torch_version":torch.__version__, "sklearn_version":sklearn.__version__,
    "numpy_version":np.__version__, "pandas_version":pd.__version__,
}
(RUN_DIR / "model_config.json").write_text(
    json.dumps(model_config, ensure_ascii=False, indent=2), encoding="utf-8")
(RUN_DIR / "validation_metrics.json").write_text(
    json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")
print("저장한 폴더:", RUN_DIR.resolve())
